# GraSCCo – LLM-based De-identification Pipeline (Processed Input) + Evaluation

This notebook contains the **improved version** of the de-identification pipeline:

- The LLM marks PII using the format `<<LABEL|ORIGINAL>>` (the original value is still visible inside the marker)
- The pipeline replaces markers deterministically with tokens such as `[LABEL_1]`
- A **mapping table** is stored (token → label + original value)
- Leakage evaluation can be performed on the produced output files

**Important concept:**  
*Pseudonymization* keeps a reversible mapping, while *anonymization* does not.


**Important:** Gold `spans` (annotations) are used **only for evaluation**.

They are **not** meant to be used as a fallback to modify the LLM output unless you explicitly enable the span-based fallback in the code.


In [ ]:

# Imports used throughout the notebook.
# Main dependencies are PyTorch + Hugging Face Transformers for running the LLM, and pandas for light data handling.


import json
import re
import gc
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

import pandas as pd


In [ ]:
# Configuration section:
# - model identifiers (Hugging Face)
# - file paths for input + outputs
# - label sets for anonymization vs pseudonymization
# only adjust this block when switching datasets, prompts, or output folders.

# CONFIG

MODELS = {
    "mistral": "mistralai/Mistral-7B-Instruct-v0.2",
    # gated: brauchst HF-Login + Access; wird unten try/except behandelt
    "llama": "meta-llama/Meta-Llama-3-8B-Instruct",
    # fallback (offen) – optional, falls du ohne gated arbeiten willst:
}

DTYPE = torch.float16

INPUT_JSON  = "data/input/grascco_input.json"
PROMPT_FILE = "final_prompt.txt"   


OUT_DIR = "data/output_processed/"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

# Outputs (combined anonymized + pseudonymized)
OUT_COMBINED = {
    "mistral": str(Path(OUT_DIR) / "grascco_mistral_combined.json"),
    "llama":   str(Path(OUT_DIR) / "grascco_llama_combined.json"),
}
# Mapping file for pseudonymization
OUT_MAPPING = {
    "mistral": str(Path(OUT_DIR) / "mapping_mistral.json"),
    "llama":   str(Path(OUT_DIR) / "mapping_llama.json"),
}

USE_ORIGINAL_TEXT = False

# Categories that are anonymized 
ANONYMIZE_LABELS = {
    "CONTACT_PHONE", "CONTACT_FAX", "CONTACT_EMAIL",
    "ID", "CASE_ID",
    "DATE_BIRTH", "DATE", "AGE",
    "LOCATION_STREET", "LOCATION_ZIP"
}

# Categories that are pseudonymised (revocable)
PSEUDONYMIZE_LABELS = {
    "NAME_PATIENT", "NAME_DOCTOR", "NAME_RELATIVE", "NAME_TITLE",
    "LOCATION_HOSPITAL", "LOCATION_CITY", "LOCATION_COUNTRY"
}

print("Config ready:", OUT_DIR)

In [ ]:
# Small helper functions for safe file I/O.
# They keep the main pipeline code cleaner (load JSON, save JSON, and check if outputs already exist).

# IO HELPERS

def safe_load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def safe_save_json(obj, path: str):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def output_exists(path: str) -> bool:
    p = Path(path)
    return p.exists() and p.is_file() and p.stat().st_size > 0


## 1) Load the system prompt and enforce marker-based output



1. Return **only** the processed German text (no JSON / explanations)
2. Wrap every detected PII span as `<<LABEL|ORIGINAL>>`
3. Keep medical content unchanged


In [ ]:
# This cell loads the base prompt from disk and appends strict rules for the marker format.
# The final SYSTEM_PROMPT is passed as the system message to the LLM.

# PROMPT LOAD + MARKER INSTRUCTIONS
with open(PROMPT_FILE, "r", encoding="utf-8") as f:
    BASE_PROMPT = f.read()

MARKER_RULES = """WICHTIG (FORMAT):
- Gib NUR den bearbeiteten deutschen Text zurück (kein JSON, keine Listen, kein Markdown, keine Erklärungen).
- KEINE Einleitung, KEINE Schritte, KEINE Kommentare.
- Der Output MUSS mit dem ersten Zeichen des Input-Texts beginnen und mit dem letzten Zeichen enden.

WICHTIG (MARKER):
- Markiere JEDE erkannte PHI/PII-Stelle exakt im Format: <<LABEL|ORIGINALTEXT>>
- Beispiele: <<NAME_PATIENT|Example Person>>, <<NAME_DOCTOR|Dr. Example Clinician>>, <<LOCATION_CITY|Example City>>, <<CONTACT_EMAIL|person@example.invalid>>
- Der Originalwert darf NUR innerhalb von <<...|...>> vorkommen.
- Bestehende Maskierungen wie [DATE], [PHONE], [ID], [EMAIL] dürfen NICHT verändert werden.
- Medizinischer Inhalt muss unverändert bleiben.
"""

SYSTEM_PROMPT = BASE_PROMPT + "" + MARKER_RULES
print(" SYSTEM_PROMPT length:", len(SYSTEM_PROMPT))

In [ ]:

# We keep the system prompt in one variable to avoid having multiple versions in the notebook.
# This makes the pipeline easier to reproduce and debug.

# Use the prompt loaded from PROMPT_FILE (regex_prompt2.txt)
# Keep ONE source of truth to avoid contradictions.
DEID_SYSTEM_PROMPT = SYSTEM_PROMPT


## 2) Deterministic tokenization + mapping

Marker output like `<<LABEL|ORIGINAL>>` is converted into tokens such as `[LABEL_1]`.

- The mapping uses the token as key → `{label, original}`
- Tokens are consistent within one run (same label + original → same token)


In [ ]:
# Marker post-processing:
# Detects patterns like <<LABEL|ORIGINAL>>
# Converts them into tokens ([LABEL] for anonymized labels, [LABEL_n] for pseudonymized labels)
# Builds a mapping table so pseudonyms remain consistent within the run

import re

MARKER_RE = re.compile(r"<<\s*([A-Z_]+)\s*\|\s*(.*?)\s*>>", re.DOTALL)

def pseudonymize_markers(text: str, mapping: dict, counters: dict):
    """
    Parse LLM output containing markers <<LABEL|ORIGINAL>> and turn them into:
      - anonymized placeholders: [LABEL]   (for labels in ANONYMIZE_LABELS)
      - pseudonymized tokens:    [LABEL_n] (for labels in PSEUDONYMIZE_LABELS)
    Only labels in the whitelist (ANONYMIZE_LABELS ∪ PSEUDONYMIZE_LABELS) are processed.
    Any unknown/hallucinated label (e.g., RR, LAB, PHONE without label in list) is ignored:
    the marker is removed and ORIGINAL text is kept to avoid medical data loss.
    Returns: (processed_text, mapping, counters, n_replacements)
    """
    allowed_labels = set(ANONYMIZE_LABELS) | set(PSEUDONYMIZE_LABELS)

    def get_token(label: str, original: str) -> str:
        key = (label, original)
       
        for tok, meta in mapping.items():
            if isinstance(meta, dict) and meta.get("label") == label and meta.get("original") == original:
                return tok
        n = counters.get(label, 0) + 1
        counters[label] = n
        return f"[{label}_{n}]"

    nrep = 0
    out_parts = []
    last = 0
    for m in MARKER_RE.finditer(text):
        out_parts.append(text[last:m.start()])
        label = (m.group(1) or "").strip()
        original = (m.group(2) or "")
        if label not in allowed_labels:
            out_parts.append(original)
        else:
            if label in ANONYMIZE_LABELS:
                out_parts.append(f"[{label}]")
                nrep += 1
            else:
                tok = get_token(label, original)
                out_parts.append(tok)
                # mapping stores ONLY pseudo labels
                mapping[tok] = {"label": label, "original": original}
                nrep += 1
        last = m.end()
    out_parts.append(text[last:])
    return "".join(out_parts), mapping, counters, nrep


def enforce_anon_vs_pseudo(text: str, mapping: dict):
    """
    Final safety gate:
    - ensure mapping contains ONLY PSEUDONYMIZE_LABELS
    - ensure no anonymized labels have numbered tokens in text
    """
    # Drop any mapping entries that are not pseudo labels
    mapping = {k: v for k, v in mapping.items()
               if isinstance(v, dict) and v.get("label") in PSEUDONYMIZE_LABELS}

    # Replace any accidentally numbered anon tokens like [CONTACT_PHONE_1] -> [CONTACT_PHONE]
    # (Shouldn't happen with pseudonymize_markers, but kept as a safety net)
    for label in ANONYMIZE_LABELS:
        text = re.sub(rf"\[{re.escape(label)}_\d+\]", f"[{label}]", text)

    return text, mapping


In [ ]:

# Final post-processing step to enforce the difference between:
#  anonymization (no reversible mapping)
#  pseudonymization (mapping stored)
# This prevents accidental reversible tokens for labels that should be anonymized.


TOKEN_RE = re.compile(r"\[([A-Z0-9_]+)_(\d+)\]")

def enforce_anon_vs_pseudo(text: str, mapping: dict):
    """
    Converts tokens for ANONYMIZE_LABELS from [LABEL_n] -> [LABEL] (no number)
    and removes them from the mapping (not reversible).
    Returns: (text_fixed, mapping_fixed)
    """
    if not text:
        return text, mapping
    if mapping is None:
        mapping = {}
    new_mapping = {}
    # replace anon tokens in text
    for tok, meta in mapping.items():
        label = (meta.get("label") or "").upper()
        if label in ANONYMIZE_LABELS:
            text = text.replace(tok, f"[{label}]")
        else:
            new_mapping[tok] = meta
    return text, new_mapping


In [ ]:

# Main pipeline function.
# For each document it:
# 1) runs the LLM on the input text
# 2) tokenizes marker output into placeholders
# 3) saves the de-identified text + mapping to disk
# The function also supports skipping runs if output files already exist.



def run_model_deidentification(model_key: str, max_docs: int | None = None):
    model_name = MODELS[model_key]
    out_path = OUT_COMBINED[model_key]
    map_path = OUT_MAPPING[model_key]
    # skip if output and mapping exist
    if output_exists(out_path) and output_exists(map_path):
        print(f"[SKIP] already exists: {out_path} + mapping")
        return
    input_data = safe_load_json(INPUT_JSON)
    if max_docs is not None:
        input_data = input_data[:max_docs]
    print(f" RUN {model_key.upper()} -> {model_name}")
    print(f"Docs: {len(input_data)}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=DTYPE, device_map='auto')
    if model.generation_config.pad_token_id is None:
        model.generation_config.pad_token_id = tokenizer.pad_token_id
    mapping = {}
    counters = {}
    results = []
    for idx, entry in enumerate(input_data):
        text_input = entry.get('text_original') if USE_ORIGINAL_TEXT else entry.get('text')
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': text_input},
        ]
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        enc = tokenizer(prompt_text, return_tensors='pt', padding=True, truncation=True, max_length=8192)
        enc = {k: v.to(model.device) for k, v in enc.items()}
        with torch.no_grad():
            out = model.generate(**enc, do_sample=False, max_new_tokens=2048, pad_token_id=tokenizer.pad_token_id)
        gen_ids = out[0][enc['input_ids'].shape[-1]:]
        decoded = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        if decoded.startswith('```'):
            decoded = decoded.strip('`').strip()
        # pseudonymize markers: replace <<LABEL|ORIGINAL>> with [LABEL_n]
        pseudo_text, mapping, counters, nrep = pseudonymize_markers(decoded, mapping, counters)
        # Enforce anon/pseudo policy: anonymized labels must NOT be stored in mapping
        pseudo_text, mapping = enforce_anon_vs_pseudo(pseudo_text, mapping)
        entry_out = dict(entry)
        entry_out['text_deidentified'] = pseudo_text
        entry_out['marker_replacements'] = nrep
        results.append(entry_out)
        if (idx + 1) % 5 == 0:
            safe_save_json(results, out_path)
            safe_save_json(mapping, map_path)
        print(f"  ✓ {idx+1}/{len(input_data)} | markers={nrep}")
    safe_save_json(results, out_path)
    safe_save_json(mapping, map_path)
    print('Saved:', out_path)
    print('Mapping:', map_path, '| entries:', len(mapping))
    del model, tokenizer
    torch.cuda.empty_cache()
    gc.collect()

## 3) LLM pipeline: mark → tokenize → save mapping

Workflow per document:
1. The LLM receives `{doc_id, text}`
2. The LLM returns the text with markers `<<LABEL|ORIGINAL>>`
3. Markers are replaced by tokens and the mapping is stored
4. Optional: if 0 markers were found, fall back to span-based replacement (if spans are available)


###  Full run 


In [ ]:
# Full run with the Mistral model.
# This processes the entire dataset (unless you set max_docs).

run_model_deidentification("mistral", max_docs=None)


In [ ]:

# Full run with the Llama model.
# This may fail if the Hugging Face repository is gated and you do not have access.

run_model_deidentification("llama", max_docs=None)

